# Práctica 2
Manuel Suano Gallardo

In [ ]:
import redis
from datetime import datetime, timezone
import pandas as pd
import numpy as np


# Conexión con Redis
bd_redis = redis.Redis(host='127.0.0.1', port=6379, password='')

# Esto lo tenía para vaciar la base de datos cada vez que hacía alguna prueba
# bd_redis.flushdb()

# Contador para el id de usuario y para los post
bd_redis.set('global:id_usuario', 0)
bd_redis.set('global:id_post', 0)

True

## 3.1. Diseño de la base de datos

La base de datos tendrá la siguiente estructura:

Para la información correspondiente a los usuarios se tendrá dos estructuras: una
para poder obtener el nombre de usuario dado un ID y otra para obtener el ID dado
un nombre de usuario.

	user:
		<id>:<nombre_usuario>
	users:
		<nombe_usuario>:<id>

Para la información de los seguidores y seguidos se usarán conjuntos ordenados, 
donde la puntuación es el timestamp. De esta forma nos aseguramos que se ordenen
según el momento en el que un usuario sigue a otro.

	followers:
		<id_usuario>:(set ordenado por timestamp)
			<id_usuario_seguidor>:<timestamp>
	following:
		<id_usuario>:(set ordenado por timestamp)
			<id_usuario_seguido>:<timestamp>

En el caso de los post se tendrá dos estructuras: una que almacenará la información
de un post en un hash asociado al ID del post y otra con los posts que se verían
en la timeline de cada usuario. En el caso de la información asociada al post, se
ha optado por un hash para almacenar los tres campos de información asociado a un
post. En el caso de los posts que se guardarán a un usuario se ha optado por un
conjunto ordenado para poder obtenerlos de más reciente a menos a la hora de 
mostrarlos en el timeline.

	post:
		<id_post>:(hash)id_usuario <id_usuario> mensaje <mensaje> timestamp <timestamp>
	posts:
		<id_usuario>:(set ordenado por timestamp)
			<id_post>:<timestamp>


Para las funciones que se definen en este apartado se ha generado una clase 
denominada OpBasicas, con dichas funciones definidas como métodos de clase. Se ha
decidido definir esta clase para ajustar la práctica al texto:
> Las funciones anteriores (nuevo_follower y nuevo_following) sólo podrán ser 
llamadas desde esta función, nunca de forma directa.

In [2]:
class OpBasicas():
	"""
	Operaciones básicas definidas en el apartado 3.1 de la práctica. 
	
	Se ha creado una clase para poder implementar el requisito de "Las funciones 
	anteriores (nuevo_follower y nuevo_following) sólo podrán ser llamadas desde
	esta función, nunca de forma directa"
	"""

	@classmethod
	def nuevo_usuario(cls, usuario: str):
		"""
		Almacena el nuevo usuario.

		Params
		------
		usuario : `str`
			Nombre del nuevo usuario que se guardará.
		
		Raises
		------
		`RuntimeError`
			Si el usuario ya existe.
		"""
		if not bd_redis.exists('users', usuario):
			id_usuario = bd_redis.incr('global:id_usuario')
			bd_redis.set(f'user:{id_usuario}', usuario)
			bd_redis.set(f'users:{usuario}', id_usuario)
		else:
			raise RuntimeError('El usuario ya existe')

	@classmethod
	def __nuevo_follower(cls, id_usuario: str, id_seguidor: str, timestamp: float):
		"""
		Añade el usuario asociado a `id_seguidor` como seguidor del usuario 
		asociado a `id_usuario`.

		Params
		------
		id_usuario : `str`
			Usuario sobre le que recae la acción de seguir.

		id_seguidor : `str`
			Usuario que realiza la acción de seguir.
		
		timestamp : `float`
			Momento en segundos en el que se produce la acción.
		"""		
		bd_redis.zadd(f'followers:{id_usuario}', {id_seguidor : timestamp})

	@classmethod
	def __nuevo_following(cls, id_usuario: str, id_seguido: str, timestamp: float):
		"""
		Añade el usuario asociado a `id_seguido` como usuario seguido por el 
		usuario asociado a `id_usuario`.

		Params
		------
		id_usuario : `str`
			ID del usuario que realiza la acción de seguir.

		id_seguido : `str`
			ID del usuario sobre el que recae la acción de seguir.
		
		timestamp : `float`
			Momento en segundos en el que se produce la acción.
		"""		
		bd_redis.zadd(f'following:{id_usuario}', {id_seguido : timestamp})

	@classmethod
	def seguir(cls, usuario: str, usuario_seguido: str, timestamp: float):
		"""
		Genera la relación de seguimiento entre `usuario` y `usuario_seguido`.

		Params
		------
		usuario : `str`
			Usuario que realizar la acción de seguir.

		usuario_seguido : `str`
			Usuario sobre el que recae la acción de seguir.

		timestamp : `float`
			Momento en segundos en el que se realiza la acción.

		Raises
		------
		`RuntimeError`
			Si `usuario` no existe en la BD.
			Si `usuario_seguido` no existe en la BD.
			Si `usuario` ya sigue a `usuario_seguido`.
			Si `usuario` y `usuario_seguido` son el mismo usuario
		"""
		try:
			if not bd_redis.exists(f'users:{usuario}'):
				raise RuntimeError(f'El usuario {usuario} no existe')
		
			if not bd_redis.exists(f'users:{usuario_seguido}'):
				raise RuntimeError(f'El usuario {usuario_seguido} no existe')
			
			if usuario == usuario_seguido:
				raise RuntimeError('Un usuario no puede seguirse a sí mismo')
			
			id_usuario = bd_redis.get(f'users:{usuario}')
			id_seguido = bd_redis.get(f'users:{usuario_seguido}')

			if bd_redis.zscore(f'following:{id_usuario}', id_seguido) is not None:
				raise RuntimeError(f"El usuario {usuario} ya sigue al usuario {usuario_seguido}")

			# En la práctica se indica que estas funciones deben recibir los 
			# nombres de usuario, pero como he extraído la lógica de las 
			# comprobaciones a esta función con el objetivo de no realizar dos 
			# veces las mismas comprobaciones, a las funciones les voy a pasar 
			# los IDs directamente.
			cls.__nuevo_follower(id_seguido, id_usuario, timestamp)
			cls.__nuevo_following(id_usuario, id_seguido, timestamp)
		except RuntimeError as re:
			print(re)
		except Exception as ex:
			print("Error no conocido:", ex)
	
	@classmethod
	def nuevo_post(cls, usuario: str, mensaje: str, timestamp: float):
		"""
		Guarda el post con contenido `mensaje` generado por `usuario` en el 
		momento `timestamp`.

		Params
		------
		usuario : `str`
			Usuario que genera el post.
		
		mensaje : `str`
			Contenido del post.

		timestamp : `float`
			Momento en segundos en el que se genera el post.
		"""
		if not bd_redis.exists(f'users:{usuario}'):
			raise RuntimeError(f'El usuario {usuario} no existe')

		if mensaje is None or len(mensaje.strip()) == 0:
			raise RuntimeError(f'El mensaje no puede ser vacío')
		
		id_usuario = bd_redis.get(f'users:{usuario}')
		id_post = bd_redis.incr('global:id_post')

		# Creación del post y guardado
		bd_redis.hset(f"post:{id_post}", mapping={
			"id_usuario" 	: id_usuario, 
			"mensaje" 		: mensaje,
			"timestamp" 	: timestamp
		})

		# Adición del post en el timeline del propio usuario
		bd_redis.zadd(f'posts:{id_usuario}', {id_post: timestamp})

		# Adición del post en el timeline de los seguidores
		for id_seguidor in bd_redis.zrange(f'followers:{id_usuario}', start=0, end=-1):
			bd_redis.zadd(f'posts:{id_seguidor}', {id_post: timestamp})

## 3.2. Conjuntos de datos a usar

Tras cargar los conjuntos de datos desde los CSV, a cada uno se le aplicará un preprocesado diferente para tener los datos en el formato que se quiera.

In [3]:
ejemplo_twitter = pd.read_csv('./twitter_sample.csv', doublequote='"', delimiter=',')
relaciones = pd.read_csv('./relations.csv', delimiter=",")

print(">> Archivo twitter_sample.csv.\n> Columnas:", ejemplo_twitter.columns, "\n> Número elementos:", len(ejemplo_twitter))
print(">> Archivo twitter_sample.csv.\n> Columnas:", relaciones.columns, "\n> Número elementos:", len(relaciones))

>> Archivo twitter_sample.csv.
> Columnas: Index(['User', 'Post_Time', 'Tweet_Content', 'Unnamed: 3'], dtype='object') 
> Número elementos: 111
>> Archivo twitter_sample.csv.
> Columnas: Index(['User', 'Follows', 'Following_Time'], dtype='object') 
> Número elementos: 22


En el caso de "twitter_sample.csv", tras cargar el archivo se puede comprobar que se añade una columna de más que no contiene ninguna información, por lo que deberá de ser eliminada. 

También se puede comprobar que existe un post para el cual no se tiene el usuario que lo publica, por lo que será eliminado. 

Por último, podemos convertir la columnas `Post_Time` a timestamp para facilitar la inserción en la base de datos.

In [4]:
ejemplo_na = ejemplo_twitter.isna()

print(ejemplo_na[ejemplo_na['User'] == True])
print(ejemplo_na[ejemplo_na['Post_Time'] == True])
print(ejemplo_na[ejemplo_na['Tweet_Content'] == True])

    User  Post_Time  Tweet_Content  Unnamed: 3
14  True      False          False        True
Empty DataFrame
Columns: [User, Post_Time, Tweet_Content, Unnamed: 3]
Index: []
Empty DataFrame
Columns: [User, Post_Time, Tweet_Content, Unnamed: 3]
Index: []


In [5]:
ejemplo_twitter = ejemplo_twitter.drop(columns=ejemplo_twitter.columns[-1])
ejemplo_twitter = ejemplo_twitter.dropna(axis='index')

# Conversión de str a datetime y a timestamp en segundos.
ejemplo_twitter['Post_Time'] = pd.to_numeric(pd.to_datetime(ejemplo_twitter['Post_Time'], format="%d %b %Y %H:%M:%S")) / 10**9 # <--- Aquí estaba el error

Por otro lado, en el caso de "relations.csv", primero vamos a aplicar la misma conversión a la columna `Following_Time`. Después, extraeremos los usuarios únicos que existen en el archivo para crearlos en la BD.

In [6]:
relaciones['Following_Time'] = pd.to_numeric(pd.to_datetime(relaciones['Following_Time'], format="%d %b %Y %H:%M:%S")) / 10**9 # <--- Aquí estaba el error

usuarios_bd = np.unique( np.append(relaciones['User'].unique(), relaciones['Follows'].unique()))

A partir de toda esta información, procedemos a insertarla en la base de datos:

In [7]:
# Primero insertamos los usuarios
for usuario in usuarios_bd:
	OpBasicas.nuevo_usuario(usuario)

# Ahora insertamos las relaciones de seguimiento
for relacion_seguimiento in relaciones.values:
    OpBasicas.seguir(relacion_seguimiento[0], relacion_seguimiento[1], relacion_seguimiento[2])

# Y por último los posts
for post in ejemplo_twitter.values:
    OpBasicas.nuevo_post(post[0], post[2], post[1])

## 3.3. Pruebas

Implementación de las tres funciones:

In [8]:
def obtener_followers(usuario: str):
	"""
	Obtiene la lista de seguidores del usuario.

	Param
	-----
	usuario : `str`
		Usuario del que se va a obtener la lista.

	Raises
	------
	`RuntimeError`
		Si el usuario no existe.
	"""
	if not bd_redis.exists(f'users:{usuario}'):
		raise RuntimeError(f'El usuario {usuario} no existe')

	# Obtenemos la lista de los ID de los seguidores
	lista_ids_timestamp = bd_redis.zrange(
		f'followers:{bd_redis.get(f'users:{usuario}')}',
		start=0,
		end=-1,
		withscores=True
	)

	# Por si no tuviera seguidores
	if lista_ids_timestamp == []:
		print(f"El usuario {usuario} no tiene seguidores")
	else:
		lista_seguidores = []

		# Lista de los seguidores pero con el nombre y la fecha en formato legible.
		for seguidor in lista_ids_timestamp:
			nombre_seguidor = bd_redis.get(f'user:{seguidor[0].decode('ascii')}')
			lista_seguidores.append((
				nombre_seguidor.decode('ascii'),
				# He añadido timezone.utc para que coincida con la fecha de "relations.csv"
				datetime.fromtimestamp(seguidor[1], timezone.utc).strftime("%d %b %Y %H:%M:%S")
			))

		# Impresión de los seguidores
		print(f">> Seguidores de {usuario}")
		for seguidor in lista_seguidores:
			print(f"> {seguidor[0]} desde el {seguidor[1]}")


def obtener_followings(usuario):
	"""
	Obtiene la lista de seguidos del usuario.

	Param
	-----
	usuario : `str`
		Usuario del que se va a obtener la lista.

	Raises
	------
	`RuntimeError`
		Si el usuario no existe.
	"""
	if not bd_redis.exists(f'users:{usuario}'):
		raise RuntimeError(f'El usuario {usuario} no existe')

	# Obtenemos la lista de los ID de los seguidos
	lista_ids_timestamp = bd_redis.zrange(
		f'following:{bd_redis.get(f'users:{usuario}')}',
		start=0,
		end=-1,
		withscores=True
	)
	
	# Por si no siguiera a ningún otro usuario
	if lista_ids_timestamp == []:
		print(f"El usuario {usuario} no sigue a nadie")
	else:
		lista_seguidos = []

		# Lista de los seguidores pero con el nombre y la fecha en formato legible.
		for seguido in lista_ids_timestamp:
			nombre_seguido = bd_redis.get(f'user:{seguido[0].decode('ascii')}')
			lista_seguidos.append((
				nombre_seguido.decode('ascii'), 
				# He añadido timezone.utc para que coincida con la fecha de "relations.csv"
				datetime.fromtimestamp(seguido[1], timezone.utc).strftime("%d %b %Y %H:%M:%S")
			))

		# Impresión de los seguidores
		print(f">> Seguidos de {usuario}")
		for seguido in lista_seguidos:
			print(f"> {seguido[0]} desde el {seguido[1]}")


def obtener_timeline(usuario: str, tweets_propios: bool):
	"""
	Obtiene el timeline de un determinado usuario.

	Params
	------
	usuario : `str`
		Nombre del usuario del que se quiere el timeline

	tweets_propios : `bool`
		Indicador para que se vean (True) o no (False) los tweets propios del 
		usuario.
	
	Raises
	------
	`RuntimeError`
		Si el usuario no existe.
	"""
	if not bd_redis.exists(f'users:{usuario}'):
		raise RuntimeError(f'El usuario {usuario} no existe')

	id_usuario = bd_redis.get(f'users:{usuario}')

	# Obtenemos la lista de los tweets ordenados. Devuelve un array con la 
	# información de los tweets de 3 en 3.
	lista_tweets_id = bd_redis.sort(
		f"posts:{id_usuario}",
		by="nosort",
		get=["post:*->id_usuario", "post:*->mensaje", "post:*->timestamp"],
		desc=True # Para que salga de más reciente a más antiguo, como realmente sería
	)

	# Obtenemos las fechas de seguimiento con los otros usuarios
	lista_seguidos_timestamp = bd_redis.zrange(
		f'following:{bd_redis.get(f'users:{usuario}')}',
		start=0,
		end=-1,
		withscores=True
	)

	# Restructuración de la lista de seguidos para más facilidad de acceso al 
	# timestamp.
	seguidos = { seguido[0].decode() : seguido[1] for seguido in lista_seguidos_timestamp }

	for indice_tweet in range(0, len(lista_tweets_id), 3):
		
		# Extraemos la info y la decodificamos en utf-8
		id_usuario_post = lista_tweets_id[indice_tweet].decode()
		post = lista_tweets_id[indice_tweet + 1].decode()
		timestamp = float(lista_tweets_id[indice_tweet + 2].decode())

		# Si es el mismo usuario y se quieren mostrar los tweets propios.
		if id_usuario.decode() == id_usuario_post:
			if tweets_propios:
				print(f">> {usuario} / {datetime.fromtimestamp(timestamp, timezone.utc).strftime("%d %b %Y %H:%M:%S")}")
				print(f"> {post}\n")
		# Si no es el mismo usuario y el momento de publicación del tweet es 
		# posterior al momento de seguimiento por parte del usuario al usuario
		# que ha publicado el tweet.
		elif timestamp >= seguidos[id_usuario_post]:
			print(f">> {bd_redis.get(f'user:{id_usuario_post}').decode()} / {datetime.fromtimestamp(timestamp, timezone.utc).strftime("%d %b %Y %H:%M:%S")}")
			print(f"> {post}\n")

Algunos ejemplos del funcionamiento de las funciones.

In [9]:
obtener_followers('animalhealthEU')

>> Seguidores de animalhealthEU
> afparron desde el 01 Jun 2019 06:04:40
> charleskod desde el 01 Jul 2019 07:27:58
> dkalnow desde el 01 Jul 2019 08:40:26
> alkhalilkouma desde el 01 Jul 2019 19:25:03


In [10]:
obtener_followings('animalhealthEU')

>> Seguidos de animalhealthEU
> drshahrul80 desde el 19 Jul 2019 14:59:55
> dkalnow desde el 27 Aug 2019 13:57:04


In [11]:
obtener_timeline('andyglittle', True)

>> andyglittle / 30 Aug 2019 09:55:43
> Another spot of our #morethanmedicine bus in #bristol this week! If you need support with your cancer diagnosis call us on 0303 3000 118. #livingwellwithcancer https://t.co/eZGLz0BkXB

>> seers_helen / 30 Aug 2019 00:49:54
> What a great team ⁦@HealthSourceOH⁩ ⁦@Local12⁩ #morethanmedicine https://t.co/g2YzMDUpVA

>> seers_helen / 29 Aug 2019 19:54:36
> Will you be at #FIX19? Want a preview of @AG_EM33 story? Then check back Monday for #ChangeOfHeart where we sat down with Alin to discuss her new life since her heart #transplant #MoreThanMedicine https://t.co/Xl9zjr7kZ1

>> seers_helen / 28 Aug 2019 05:18:35
> Czy wiesz, że do bez leków dla zwierząt potrzeba by 89% więcej zwierząt do wyprodukowania tej samej ilości mleka i wołowiny?
#MoreThanMedicine https://t.co/SEtHxlV4p1

>> andyglittle / 27 Aug 2019 16:07:40
> For nearly 40 years @PennyBrohnUK has been a link between standard medical treatment and social prescribing, that's why we're at the se

## Impresiones finales

La práctica me ha gustado mucho porque presenta un caso real, aunque simplificado, que creo que ayuda bastante a entender cómo funciona Redis. A mí, personalmente, el material teórico que se proporciona no me terminaba de dejar claro los tipos de datos y como se podían usar, especialmente el hash, pero al realizar la práctica siento que lo entiendo todo mucho mejor.

Sin embargo, creo que podría haber sido útil si en la parte teórica se hubiese añadido algunas diapositivas con la información básica de cómo realizar las diferentes operaciones en Python, ya que, aunque es fácil una vez se le "pilla el truco", al principio se hace un poco complicado de entender qué se está haciendo realmente, más cuando la documentación oficial tampoco ofrece ejemplos en Python.